# ERA5 reconstruction

The supplied ERA5 arrays are already standardized. The source demonstration observes 5% of positions in all four channels and uses 50 reverse steps. The evaluation notebooks include 2% and the full observation-rate sweep.

Run cells in order. Sampling counts are reverse-diffusion steps; checkpoint numbers identify training state.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'models').is_dir() and (p / 'sampler').is_dir() and (p / 'trainer').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the MRSM repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT / 'outputs' / 'era5'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

def format_current_figure():
    def tick(value, pos):
        return '0' if abs(value) < 1e-12 else f'{value:g}'
    for ax in plt.gcf().axes:
        if ax.get_xscale() == 'linear':
            ax.xaxis.set_major_formatter(FuncFormatter(tick))
        if ax.get_yscale() == 'linear':
            ax.yaxis.set_major_formatter(FuncFormatter(tick))

print('Repository:', ROOT)
print('Device:', device)


In [ ]:
import os
from sampler.utils import *
import importlib
import sampler.utils
from trainer.utils import restore_checkpoint
import torch
import torch.nn as nn
import json
from collections import namedtuple
from tqdm import tqdm
from models import get_model, get_ema
from sampler import VESDE
import numpy as np
from einops import rearrange
data_name = 'era5'
version = 'v0'
opt_path = './results/' + data_name + '_' + version + '/config.json'
with open(opt_path, 'r', encoding='utf-8') as f:
    opt = json.load(f)
    opt['num_channels'] = opt['num_components'] + opt['num_conditions']
opt['results_path'] = str(Path(opt_path).parent)
opt['device'] = device
OPT_class = namedtuple('OPT_class', opt.keys())
opt = OPT_class(**opt)
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
net = get_model(opt)
net.to(device)
ema = get_ema(net.parameters(), decay=opt.ema_rate)
optimizer = torch.optim.Adam(net.parameters(), lr=opt.lr, betas=(0.9, 0.999), eps=1e-08, weight_decay=0)
state = dict(optimizer=optimizer, model=net, ema=ema, epoch=0, iteration=0, loss_train=[])
ckpt_path = opt.results_path + '/checkpoint.pth'
if not Path(ckpt_path).is_file():
    raise FileNotFoundError(ckpt_path)
state = restore_checkpoint(ckpt_path, state, device)
net = state['model']
ema = state['ema']
ema.copy_to(net.parameters())
print('Model loaded from epoch', state['epoch'])


In [ ]:
data_test_path = './data/era5/era5_test.npy'
scalar_path = opt.results_path + '/scalars_era5.npy'
data_raw = np.load(data_test_path)
(_, z) = np.meshgrid(np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), indexing='ij')
x_test = data_raw.transpose(0, 1, 4, 2, 3)
print(x_test.shape)


In [ ]:
m = 2
T = opt.num_frames
T_prime = 100
B = int(np.ceil((T_prime - m) / (T - m)))
T_prime = B * (T - m) + m
n_samples = 1
samples = []
seed_samples = 1
rnd = np.random.RandomState(seed_samples)
t_start = 0
samples = x_test[:, t_start:t_start + T_prime]
print(samples.shape)


In [ ]:
outer_loop = 50
denoising_steps = 50
seed_sample = 123
obs_type = 'random'
scale = 0.95
alpha_case = 0.3
beta = 0.4
gamma = 0.5
sde = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=outer_loop)
sde1 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps)
net.eval()
reap_fn = lambda x, num: torch.stack([x for _ in range(num)])
setup_seed(seed_sample)
with torch.no_grad():
    if obs_type.lower() == 'random':
        T_prime_y = T_prime
        mask = np.zeros([opt.num_components + opt.num_conditions, opt.image_size, opt.image_size])
        spatial_mask = mask_gen([1, opt.image_size, opt.image_size], scale, seed=seed_samples)
        mask[:opt.num_components] = mask_gen([1, opt.image_size, opt.image_size], scale, seed=seed_samples)
        mask_torch = torch.from_numpy(mask).to(device)
        transform_gen = transform = lambda x: x[:, :T_prime_y] * mask[np.newaxis, np.newaxis] if isinstance(x, np.ndarray) else x[:, :T_prime_y] * mask_torch[None, None].to(x.device)
        sparsity = scale
    elif obs_type.lower() == 'regular':
        T_prime_y = T_prime
        transform_gen = transform = lambda x: x[:, :T_prime_y, :, ::scale, ::scale]
        sparsity = 1 - 1 / scale ** 2
    elif obs_type.lower() == 'forecast':
        T_prime_y = 0
        transform_gen = transform = lambda x: x[:, :scale]
        sparsity = 1 - scale / T
    alpha = alpha_case / np.sqrt(1 - sparsity)
    samples_scale = samples.copy()
    y = transform_gen(samples_scale)
    print(y.shape)
    latent_mask = torch.ones([T, 1, 1, 1]).float().to(device)
    obs_mask = torch.zeros([T, 1, 1, 1]).float().to(device)
    frame_indices = torch.arange(T).to(device)
    net_fn = lambda x, t: net(x, x0=x, timesteps=t, latent_mask=reap_fn(latent_mask, len(x)), obs_mask=reap_fn(obs_mask, len(x)), frame_indices=reap_fn(frame_indices, len(x)))[0]
    x_inpainted = s3gm_sample_2d(opt, net_fn, sde, y, transform_gen, alpha=0.2 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
    setup_seed(seed_sample)
    x_inpainted1 = s3gm_sample_2d_ISS_ERA5(opt, net_fn, sde1, y, transform_gen, alpha=0.3 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, n_steps=0, eta=0, reg_coef_pa=0, reg_coef_ar=0, denoising_steps=denoising_steps, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
x_pred = x_inpainted
x_pred1 = x_inpainted1
print(x_pred.shape)
errors = cal_rmse(samples[0], x_pred[0], normalize=True)
mean_error = np.mean(errors)
print(f'mean_error: {mean_error}')
errors1 = cal_rmse(samples[0], x_pred1[0], normalize=True)
mean_error1 = np.mean(errors1)
print(f'mean_error1: {mean_error1}')
num_T = x_pred.shape[1]
print(f'Number of generated snapshots: {num_T}')
torch.cuda.empty_cache()


## Fields and per-channel errors


In [ ]:
rows = []
(fig, axes) = plt.subplots(4, 3, figsize=(10, 10), constrained_layout=True)
for (ch, name) in enumerate(['u', 'v', 'temperature', 'pressure']):
    (lo, hi) = (np.min(samples[0, 0, ch]), np.max(samples[0, 0, ch]))
    for (col, (method, field)) in enumerate([('Reference', samples), ('SDE', x_pred), ('ISS', x_pred1)]):
        im = axes[ch, col].imshow(field[0, 0, ch], vmin=lo, vmax=hi, cmap='RdBu_r', origin='lower')
        axes[ch, col].set_title(f'{name}: {method}')
        if col:
            err = cal_rmse(samples[0, :, ch:ch + 1], field[0, :, ch:ch + 1], normalize=True)
            rows.append({'method': method, 'channel': name, 'nrmse': float(np.mean(err))})
    fig.colorbar(im, ax=axes[ch, :], label='Standardized value')
format_current_figure()
fig.savefig(OUTPUT / 'era5_fields.pdf', bbox_inches='tight')
format_current_figure()
plt.show()
pd.DataFrame(rows).to_csv(OUTPUT / 'channel_metrics.csv', index=False)
